# Day 7 · 回归指标、最终评估与结果解释

目标：理解四个指标，冻结方案，报告最终测试结果及局限。
基础 40 分钟、深度 35 分钟、广度 25 分钟、作业 40 分钟；Bonus 30 分钟。

### 1. 四个指标

令 $e_i=y_i-\hat y_i$：
- MAE = $\frac1n\sum|e_i|$，单位为分，描述平均绝对误差。
- MSE = $\frac1n\sum e_i^2$，单位为分²，对大误差更敏感。
- RMSE = $\sqrt{MSE}$，单位为分。
- R² = $1-\frac{\sum e_i^2}{\sum(y_i-\bar y)^2}$，比较评估集自身均值基准。

前三者越小越好；R² 最好为 1，也可以为负。R² 不是“预测正确百分比”。
评估标签恒定时公式分母为零；sklearn 默认对这种情况做有限值处理，解释前应检查目标变化。

In [ ]:
from pathlib import Path
import numpy as np
import pandas as pd
from IPython.display import display
from sklearn.model_selection import train_test_split
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents]
            if (p / "assignments/bootcamp/week1/data/raw/student-mat.csv").exists())
raw = pd.read_csv(ROOT / "assignments/bootcamp/week1/data/raw/student-mat.csv", sep=";")
# 固定划分，第三至七天始终使用相同学生；实际行数为 276/59/60。
train, heldout = train_test_split(raw, test_size=0.30, random_state=42)
val, test = train_test_split(heldout, test_size=0.50, random_state=42)
train, val, test = train.copy(), val.copy(), test.copy()
print("训练/验证/测试：", len(train), len(val), len(test))


In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns
sns.set_theme(style="whitegrid")


In [ ]:
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
actual = np.array([3.,-0.5,2.,7.])
predicted = np.array([2.5,0.,2.,8.])
error = actual - predicted
print("MAE:", np.abs(error).mean())
print("MSE:", (error**2).mean())
print("RMSE:", np.sqrt((error**2).mean()))
print("R2:", 1 - (error**2).sum()/((actual-actual.mean())**2).sum())
assert np.isclose(mean_absolute_error(actual,predicted),0.5)
assert np.isclose(mean_squared_error(actual,predicted),0.375)
print("很差的预测 R2:", r2_score(actual, np.full(4,100.)))

### 2. 验证集选择规则

教学主线固定两个候选：仅 G2、G1/G2/studytime/failures。
规则：验证 MAE 最低者胜出。验证分数用于选择，不是最终泛化估计。
如果你已在练习中查看过最终测试结果，继续修改方案后的分数只能作为探索结果，需新数据独立验证。

In [ ]:
from sklearn.pipeline import Pipeline
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.dummy import DummyRegressor
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score
features = ["G1", "G2", "studytime", "failures"]
target = "G3"
X_train, y_train = train[features], train[target]
X_val, y_val = val[features], val[target]
def make_model():
    return Pipeline([("imputer", SimpleImputer(strategy="median")),
                     ("regressor", LinearRegression())])
def metrics(y_true, y_pred):
    return {"MAE": mean_absolute_error(y_true, y_pred),
            "MSE": mean_squared_error(y_true, y_pred),
            "RMSE": np.sqrt(mean_squared_error(y_true, y_pred)),
            "R2": r2_score(y_true, y_pred)}
baseline = DummyRegressor(strategy="mean").fit(X_train, y_train)
model = make_model().fit(X_train, y_train)
val_pred = model.predict(X_val)
display(pd.DataFrame({
    "baseline": metrics(y_val, baseline.predict(X_val)),
    "linear": metrics(y_val, val_pred)}).T)


In [ ]:
# 两个预先规定的候选方案；选择标准仅为验证集 MAE。
feature_sets = {"G2_only": ["G2"], "multi": ["G1","G2","studytime","failures"]}
candidates = {}
rows = []
for name, cols in feature_sets.items():
    fitted = make_model().fit(train[cols], train["G3"])
    candidates[name] = fitted
    rows.append({"model": name, **metrics(val["G3"], fitted.predict(val[cols]))})
validation_results = pd.DataFrame(rows).set_index("model")
display(validation_results)
chosen = validation_results["MAE"].idxmin()
chosen_features = feature_sets[chosen]
print("冻结方案：", chosen, chosen_features)


### 3. 冻结后，最终测试

合并训练与验证，重新拟合选定 Pipeline；测试集仅预测和评价。
只运行一次完整流程形成报告。为了重现而重复运行相同冻结方案，不等于可以根据测试分数反复调参。

In [ ]:
# 选择完后，用训练+验证重新拟合；包括填补统计量。
development = pd.concat([train, val])
final_model = make_model().fit(development[chosen_features], development["G3"])
final_baseline = DummyRegressor(strategy="mean").fit(
    development[chosen_features], development["G3"])
test_pred = final_model.predict(test[chosen_features])
test_scores = pd.DataFrame({
    "baseline": metrics(test["G3"], final_baseline.predict(test[chosen_features])),
    "selected_linear": metrics(test["G3"], test_pred)}).T
display(test_scores)
# 评估后不再依据测试分数改特征、改参数或换随机种子。


### 4. 残差诊断

残差 = 实际值 − 预测值。正残差表示低估，负残差表示高估。
理想散点接近对角线；残差若随预测值出现曲线或扩散，提示模型形式或方差假设可能不适合。
检查最大误差样本用于描述局限，不能观察之后删除它们再报告“更好的测试分数”。

In [ ]:
errors = pd.DataFrame({"actual": test["G3"], "predicted": test_pred}, index=test.index)
errors["residual"] = errors["actual"] - errors["predicted"]
errors["absolute_error"] = errors["residual"].abs()
fig, axes = plt.subplots(1,2,figsize=(11,4))
axes[0].scatter(errors["actual"], errors["predicted"], alpha=0.7)
lo = min(errors["actual"].min(), errors["predicted"].min())
hi = max(errors["actual"].max(), errors["predicted"].max())
axes[0].plot([lo,hi],[lo,hi],"r--")
axes[0].set(xlabel="Actual", ylabel="Predicted", title="Final held-out predictions")
axes[1].scatter(errors["predicted"], errors["residual"], alpha=0.7)
axes[1].axhline(0,color="r",linestyle="--")
axes[1].set(xlabel="Predicted", ylabel="Actual - predicted", title="Residuals")
fig.tight_layout()
plt.show()
display(errors.nlargest(5, "absolute_error"))


In [ ]:
print("测试中范围外预测数：", int(((test_pred < 0) | (test_pred > 20)).sum()))
print("平均残差：", errors["residual"].mean())
output = ROOT / "assignments/bootcamp/week1/outputs"
output.mkdir(exist_ok=True)
test_scores.to_csv(output / "day7_test_metrics.csv")
errors.to_csv(output / "day7_test_predictions.csv", index_label="source_row")

### 5. 深度与广度

同一 MAE 可能对应完全不同的误差分布。RMSE 比 MAE 大很多时，关注少数大误差。
单次拆分样本少，结果会波动；不要筛选随机种子来获取好分数。
测试 R² 的均值基准使用测试标签均值，只是指标定义；DummyRegressor 使用开发集均值，二者不是完全相同的常数。

模型只评估来自同一数据来源的随机留出学生，不能证明跨学校或跨年份有效。
主线使用既往阶段成绩；不能称为开学时成绩预测。

### DAY 7 ASSIGNMENT

基础：手算上面 4 个样本的 MAE/MSE/RMSE/R²，并核对。
深度：解释 RMSE 的单位和 R² 为负的含义。
广度：报告基准与所选模型全部四项指标，解释最大的五个误差。
Bonus：对固定测试预测做配对 bootstrap，观察 MAE 不确定性。它只反映给定模型及当前测试样本的重采样变化，不包含训练和选模的不确定性。
交付：不少于五条有证据的结论；至少三条局限；一个未来改进方向（不在当前测试上验证）。

In [ ]:
# 在这里填写报告或添加 Markdown 单元。


### 参考答案 / Bonus

In [ ]:
rng = np.random.default_rng(42)
absolute_errors = errors["absolute_error"].to_numpy()
resampled = rng.choice(absolute_errors, size=(2000,len(absolute_errors)), replace=True).mean(axis=1)
print("条件于当前模型的 MAE bootstrap 95% 区间：", np.quantile(resampled,[0.025,0.975]))
print("测试 MAE 改善：",
      test_scores.loc["baseline","MAE"] - test_scores.loc["selected_linear","MAE"])

### 报告模板（填写）

1. 我预测什么、何时预测：
2. 清洗规则和保留的异常观测：
3. 训练集探索发现：
4. 预设候选与选择规则：
5. 测试指标及与基准比较：
6. 哪类记录误差最大：
7. 时间、样本和因果解释的限制：
8. 下次用新数据验证的改进想法：